# Chapter 04：Unary / Binary Elementwise Ops

**目标**：从 vector add 推广到 square、ReLU，并把 add 与 ReLU 融合。核心概念是通用逐元素结构和 kernel fusion。

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if ROOT.name.startswith("chapter_"):
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import torch
import triton
import triton.language as tl

from common.benchmark import bench
from common.check import assert_close
from common.utils import get_device, set_seed

device = get_device()
set_seed(0)

## 1. PyTorch reference

square 和 ReLU 是 unary op；`relu(x + y)` 组合了 binary add 与 unary ReLU。

In [ ]:
x = torch.randn(1_000_003, device=device)
y = torch.randn_like(x)
square_ref = x * x
relu_ref = torch.relu(x)
add_relu_ref = torch.relu(x + y)

## 2. Square 与 ReLU kernel

它们共用相同的 program/offset/mask 模板，只改变寄存器中的计算。Sigmoid 或 tanh 也可以沿用这个结构。

In [ ]:
@triton.jit
def square_kernel(x_ptr, output_ptr, n_elements, BLOCK_SIZE: tl.constexpr):
    offsets = tl.program_id(0) * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)
    mask = offsets < n_elements
    values = tl.load(x_ptr + offsets, mask=mask)
    tl.store(output_ptr + offsets, values * values, mask=mask)

@triton.jit
def relu_kernel(x_ptr, output_ptr, n_elements, BLOCK_SIZE: tl.constexpr):
    offsets = tl.program_id(0) * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)
    mask = offsets < n_elements
    values = tl.load(x_ptr + offsets, mask=mask)
    tl.store(output_ptr + offsets, tl.maximum(values, 0.0), mask=mask)

## 3. Fused add + ReLU

分开执行通常会写出 `x + y` 中间 tensor，再由 ReLU 读回。融合 kernel 在寄存器中完成 add 和 ReLU，只写一次最终结果。

In [ ]:
@triton.jit
def add_relu_kernel(x_ptr, y_ptr, output_ptr, n_elements, BLOCK_SIZE: tl.constexpr):
    offsets = tl.program_id(0) * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)
    mask = offsets < n_elements
    x_values = tl.load(x_ptr + offsets, mask=mask)
    y_values = tl.load(y_ptr + offsets, mask=mask)
    tl.store(output_ptr + offsets, tl.maximum(x_values + y_values, 0.0), mask=mask)

## 4. Wrapper functions

In [ ]:
def _unary(kernel, x):
    if x.ndim != 1 or not x.is_cuda or not x.is_contiguous():
        raise ValueError("expected a contiguous 1D CUDA tensor")
    output = torch.empty_like(x)
    if x.numel() > 0:
        kernel[(triton.cdiv(x.numel(), 256),)](x, output, x.numel(), BLOCK_SIZE=256)
    return output

def square(x):
    return _unary(square_kernel, x)

def relu(x):
    return _unary(relu_kernel, x)

def add_relu(x, y):
    if x.shape != y.shape or x.ndim != 1 or not x.is_cuda or not y.is_cuda:
        raise ValueError("x and y must be matching 1D CUDA tensors")
    output = torch.empty_like(x)
    if x.numel() > 0:
        add_relu_kernel[(triton.cdiv(x.numel(), 256),)](
            x, y, output, x.numel(), BLOCK_SIZE=256
        )
    return output

## 5. Correctness check

In [ ]:
assert_close("square", square(x), square_ref)
assert_close("relu", relu(x), relu_ref)
assert_close("add + relu", add_relu(x, y), add_relu_ref)

## 6. Benchmark：分开执行 vs fused

In [ ]:
print(f"PyTorch add then ReLU: {bench(lambda: torch.relu(x + y)):.3f} ms")
print(f"Triton fused add+ReLU: {bench(lambda: add_relu(x, y)):.3f} ms")

## 小结与练习

Fusion 的主要收益来自减少中间 tensor 和全局内存读写，不是减少数学运算。

**练习**：参照 square kernel 写一个 `negate` kernel，并与 `-x` 比较。